# Model check
Train a simple classifier on the gold feature and label stores, to confirm that they can feed a model and to see what leakage would do to the results

## 1. Load and join the stores

In [1]:
import pandas as pd
import pyspark
import pyspark.sql.functions as F
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATAMART = "../datamart"
labels = spark.read.parquet(f"{DATAMART}/gold/label_store/*").toPandas()
features = spark.read.parquet(f"{DATAMART}/gold/feature_store/*").toPandas()

data = labels.merge(features.rename(columns={"snapshot_date": "loan_start_date"}), on=["Customer_ID", "loan_start_date"], how="inner")
data["loan_start_date"] = pd.to_datetime(data["loan_start_date"])
print(data.shape, "bad rate:", round(data["label"].mean(), 3))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 13:58:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/28 13:58:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/28 13:58:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


(12500, 72) bad rate: 0.288


**Findings:**
- 12500 loans have both features and a label
- Bad rate is 28.8%

## 2. Split by application month

In [2]:
in_time = data[data["loan_start_date"] <= "2024-06-01"]
oot = data[data["loan_start_date"] > "2024-06-01"]
train, test = train_test_split(in_time, test_size=0.2, random_state=88, stratify=in_time["label"])

for name, df in [("train", train), ("test", test), ("oot", oot)]:
    print(f"{name}: {len(df)} loans, bad rate {df['label'].mean():.3f}, "
          f"applications {df['loan_start_date'].min():%Y-%m} to {df['loan_start_date'].max():%Y-%m}, "
          f"with clickstream {df['fe_1_avg'].notna().mean():.0%}")

train: 7179 loans, bad rate 0.289, applications 2023-01 to 2024-06, with clickstream 100%
test: 1795 loans, bad rate 0.289, applications 2023-01 to 2024-06, with clickstream 100%
oot: 3526 loans, bad rate 0.287, applications 2024-07 to 2025-01, with clickstream 0%


**Findings:**
- Train (7,179) and test (1,795) come from applications 2023-01 to 2024-06;
  the out-of-time (OOT) set is the 3,526 applications from 2024-07 to 2025-01
- The bad rate is about 28.8% across all sets
- The OOT boundary is placed at the clickstream coverage cutoff (seen in notebooks/02_data_quality.ipynb), so every in-time loan has clickstream and no OOT loan does

## 3. Fit on the training data only

In [3]:
feature_columns = [c for c in features.columns if c not in ("Customer_ID", "snapshot_date")]
without_clickstream = [c for c in feature_columns if not c.startswith("fe_")]

def check(columns, train, test, oot):
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(train[columns], train["label"])
    return {name: round(roc_auc_score(df["label"], model.predict_proba(df[columns])[:, 1]), 3)
            for name, df in [("train", train), ("test", test), ("oot", oot)]}

results = {
    "all features": check(feature_columns, train, test, oot),
    "without clickstream": check(without_clickstream, train, test, oot),
}
pd.DataFrame(results).T

,train,test,oot
all features,0.874,0.875,0.767
without clickstream,0.783,0.771,0.768


**Findings:**
- The stores successfully train and predict. The imputer and scaler are fitted on the training rows only.
- With all features, test AUC is 0.875 against 0.874 on train, so there is no sign of overfitting.
- Clickstream raises test AUC from 0.771 to 0.875, but on OOT both models score about the same (0.767 and 0.768) — the imputer fills all 20 clickstream columns with the same training-median value for every OOT row, so they carry no distinguishing signal there.

## 4. Testing what temporal leakage would look like

In [4]:
all_months = spark.read.parquet(f"{DATAMART}/silver/clickstream/*").groupBy("Customer_ID").agg(*[F.avg(f"fe_{i}").alias(f"fe_{i}_all_months") for i in range(1, 21)]).toPandas()

def add_all_months(df):
    return df.merge(all_months, on="Customer_ID", how="left")

leaky_columns = without_clickstream + [f"fe_{i}_all_months" for i in range(1, 21)]
results["leaky: clickstream over all 24 months"] = check(
    leaky_columns, add_all_months(train), add_all_months(test), add_all_months(oot))
pd.DataFrame(results).T

,train,test,oot
all features,0.874,0.875,0.767
without clickstream,0.783,0.771,0.768
leaky: clickstream over all 24 months,0.976,0.973,0.761


,train,test,oot
all features,0.874,0.875,0.767
without clickstream,0.783,0.771,0.768
leaky: clickstream over all 24 months,0.976,0.973,0.761


**Findings:**
- Averaging clickstream over all 24 months, including months after the application, raises test AUC to 0.973 -> this increase comes from future information
- On OOT the leaky model scores 0.761, no better than the model without clickstream
- This is why the feature store uses only clickstream months up to the application